In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [ ]:
df_country_name_mapping = pd.read_excel(
    Path(
        "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/Aggregation111Power_trump.xlsx"
    ),
    sheet_name="Mapping_table_country",
    header=1,
).drop(columns=["Label"])
code_map = dict(zip(df_country_name_mapping["Code"], df_country_name_mapping["Aggregation Code"]))

This df is created in the notebook `data_formatting.ipynb`

In [ ]:
df_MIRAGE = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data.parquet"
).drop(columns="Quantity")

In [ ]:
df = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data.parquet"
).drop(columns="Quantity")

exp_ok = df["Exporter"].map(code_map)
imp_ok = df["Importer"].map(code_map)

# --- Perte d'EXPORT par région finale (region de l'Importer qui a matché) ---
imp_region = imp_ok
exp_lost = (
    df[imp_region.notna() & exp_ok.isna()]
    .groupby(imp_region[imp_region.notna() & exp_ok.isna()])["k$"]
    .sum()
)
exp_total = df[imp_region.notna()].groupby(imp_region[imp_region.notna()])["k$"].sum()
pct_exp_lost = (exp_lost / exp_total * 100).sort_values(ascending=False)

# --- Perte d'IMPORT par région finale (region de l'Exporter qui a matché) ---
exp_region = exp_ok
imp_lost = (
    df[exp_region.notna() & imp_ok.isna()]
    .groupby(exp_region[exp_region.notna() & imp_ok.isna()])["k$"]
    .sum()
)
imp_total = df[exp_region.notna()].groupby(exp_region[exp_region.notna()])["k$"].sum()
pct_imp_lost = (imp_lost / imp_total * 100).sort_values(ascending=False)

# print("% k$ avec un exportateur hors matching (vue côté importeur) :\n", pct_exp_lost)
# print("\n% k$ avec un importateur hors matching (vue côté exporteur) :\n", pct_imp_lost)


In [ ]:
df_MIRAGE["Exporter"] = df_MIRAGE["Exporter"].map(code_map)
df_MIRAGE["Importer"] = df_MIRAGE["Importer"].map(code_map)

df_MIRAGE.dropna(subset=["Exporter", "Importer"], inplace=True)

## Market share

In [ ]:
df_grouped = df_MIRAGE.groupby(["Year", "MIRAGE_code"])["k$"].sum().reset_index()

df_all = pd.merge(
    left=df_MIRAGE,
    right=df_grouped,
    on=["Year", "MIRAGE_code"],
    how="left",
    suffixes=("", "_total"),
)

df_all["market_share"] = df_all["k$"] / df_all["k$_total"]

df_all = (
    df_all.copy()
    .groupby(["Year", "MIRAGE_code", "Exporter"])
    .agg({"k$": "sum", "market_share": "sum"})
    .reset_index()
)
df_all.sort_values(["Year", "Exporter", "MIRAGE_code"], inplace=True)
df_all


In [ ]:
df_all.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data_cleaned.csv",
    index=False,
)


## Market share progression

In [ ]:
df_MIRAGE["Year"].unique()

In [ ]:
# Regroupement des montant par secteur
agg = df_MIRAGE.groupby(["Exporter", "MIRAGE_code", "Year"], as_index=False)["k$"].sum()

# Rotation pour avoir une colonne par année
wide = agg.pivot(index=["Exporter", "MIRAGE_code"], columns="Year", values="k$")

years = list(range(2019, 2025))
result = pd.DataFrame(index=wide.index)
for y1, y2 in zip(years[:-1], years[1:]):
    result[f"évolution_{y1}_{y2}"] = wide[y2] / wide[y1] - 1 # ratio année N+1 et année N
result["évolution_2019_2024"] = wide[2024] / wide[2019] - 1 # ratio 2024 / 2019

result = result.reset_index().rename(columns={"Exporter": "Pays"})

In [ ]:
result.to_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/market_share_evolution.csv", index=False)

### Analysis

In [ ]:
agg_all = df_MIRAGE.groupby(["MIRAGE_code", "Year"], as_index=False)["k$"].sum()

# Rotation pour avoir une colonne par année
wide_all = agg_all.pivot(index=["MIRAGE_code"], columns="Year", values="k$")

years = list(range(2019, 2025))
result_all = pd.DataFrame(index=wide_all.index)
for y1, y2 in zip(years[:-1], years[1:]):
    result_all[f"évolution_{y1}_{y2}"] = (
        wide_all[y2] / wide_all[y1] - 1
    )  # ratio année N+1 et année N
result_all["évolution_2019_2024"] = wide_all[2024] / wide_all[2019] - 1  # ratio 2024 / 2019

result_all = result_all.reset_index().rename(columns={"Exporter": "Pays"})
result_all[["MIRAGE_code", "évolution_2019_2024"]].sort_values(
    by="évolution_2019_2024", ascending=False
)